# Thực nghiệm chấm điểm rủi ro dựa trên luật

Notebook xây dựng hai bộ Rule-based Scoring độc lập cho hai nguồn dữ liệu. Mỗi bộ luật trả về điểm rủi ro, mức cảnh báo, nguyên nhân và khuyến nghị.

Nguyên tắc thực nghiệm:

- Chỉ sử dụng tập huấn luyện để kiểm tra căn cứ của các điều kiện luật và tính giá trị điền thiếu.
- Chỉ sử dụng tập xác thực để chọn ngưỡng cảnh báo.
- Tập kiểm thử chỉ được dùng để báo cáo kết quả cuối cùng.
- Hai bộ luật sử dụng đúng các `row_id` đã dùng trong thực nghiệm Machine Learning.
- `prediction_score` là điểm luật đã chuẩn hóa về 0–1, không phải xác suất thống kê.

Giới hạn cần trình bày trong báo cáo: Nguồn 1 sử dụng kết quả học kỳ 2 nên hệ thống chỉ có thể cảnh báo sau khi các dữ liệu này đã phát sinh, không phải ngay khi sinh viên nhập học. Nguồn 2 là dữ liệu mô phỏng nên bộ luật cần được kiểm chứng lại trên dữ liệu thực tế trước khi triển khai.

## Cấu hình môi trường và đường dẫn

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Hiển thị đầy đủ bảng kết quả để thuận tiện kiểm tra.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Hỗ trợ chạy notebook từ thư mục notebooks hoặc thư mục gốc của kho mã.
root_candidates = [Path(".."), Path(".")]
PROJECT_ROOT = next(
    (path.resolve() for path in root_candidates if (path / "outputs" / "splits").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy outputs/splits. Hãy chạy notebook chuẩn bị dữ liệu trước."
    )

SPLIT_DIR = PROJECT_ROOT / "outputs" / "splits"
PREDICTION_DIR = PROJECT_ROOT / "outputs" / "predictions"
RULE_DIR = PROJECT_ROOT / "outputs" / "rules"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
RULE_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục dự án:", PROJECT_ROOT)
print("Thư mục dữ liệu đã chia:", SPLIT_DIR)

## Nạp các tập dữ liệu cố định

Notebook không tự chia lại dữ liệu. Mỗi tập được đối chiếu với file ánh xạ chính thức để tránh việc Rule-based và Machine Learning đánh giá trên các sinh viên khác nhau.

In [ ]:
def load_fixed_splits(file_prefix):
    """Đọc ba tập và kiểm tra row_id với file ánh xạ chính thức."""

    split_map = pd.read_csv(SPLIT_DIR / f"{file_prefix}_split.csv")
    assert split_map["row_id"].is_unique, f"{file_prefix}: row_id bị trùng."

    frames = {}
    for split_name in ["train", "validation", "test"]:
        # Đọc dữ liệu đã chia và kiểm tra mọi dòng đều thuộc đúng tập.
        frame = pd.read_csv(SPLIT_DIR / f"{file_prefix}_{split_name}.csv")
        expected_ids = set(
            split_map.loc[split_map["split"] == split_name, "row_id"]
        )
        assert frame["row_id"].is_unique
        assert set(frame["row_id"]) == expected_ids, (
            f"{file_prefix}/{split_name}: row_id không khớp file ánh xạ."
        )
        frames[split_name] = frame

    return frames


SOURCE_FRAMES = {
    "student_dropout_and_success": load_fixed_splits(
        "student_dropout_and_success"
    ),
    "student_dropout": load_fixed_splits("student_dropout"),
}

for source_name, frames in SOURCE_FRAMES.items():
    sizes = {split_name: len(frame) for split_name, frame in frames.items()}
    print(source_name, sizes)

## Xử lý giá trị thiếu không gây rò rỉ dữ liệu

Các giá trị dùng để điền thiếu chỉ được tính từ tập huấn luyện. Nguồn 1 hiện không có giá trị thiếu trong các trường dùng cho luật; Nguồn 2 có thể thiếu `Stress_Index` và `Study_Hours_per_Day`.

In [ ]:
RULE_INPUTS = {
    "student_dropout_and_success": {
        "numeric": [
            "Curricular units 1st sem (enrolled)",
            "Curricular units 1st sem (approved)",
            "Curricular units 2nd sem (enrolled)",
            "Curricular units 2nd sem (approved)",
            "Curricular units 2nd sem (grade)",
            "Curricular units 2nd sem (without evaluations)",
            "Tuition fees up to date",
            "Debtor",
        ],
        "categorical": [],
        "max_score": 11,
    },
    "student_dropout": {
        "numeric": [
            "GPA",
            "Attendance_Rate",
            "Stress_Index",
            "Study_Hours_per_Day",
            "Assignment_Delay_Days",
        ],
        "categorical": ["Internet_Access", "Part_Time_Job"],
        "max_score": 11,
    },
}


def fit_imputation_values(train_frame, numeric_columns, categorical_columns):
    """Tính trung vị và giá trị phổ biến nhất chỉ từ tập huấn luyện."""

    values = {}
    for column in numeric_columns:
        values[column] = float(train_frame[column].median())
    for column in categorical_columns:
        mode = train_frame[column].mode(dropna=True)
        if mode.empty:
            raise ValueError(f"Không thể xác định giá trị điền thiếu cho {column}.")
        values[column] = str(mode.iloc[0])
    return values


def apply_imputation(frame, imputation_values):
    """Áp dụng các giá trị đã học từ train cho một tập dữ liệu."""

    prepared = frame.copy()
    for column, value in imputation_values.items():
        prepared[column] = prepared[column].fillna(value)
    return prepared


IMPUTATION_VALUES = {}
PREPARED_FRAMES = {}
for source_name, config in RULE_INPUTS.items():
    frames = SOURCE_FRAMES[source_name]
    # Mọi thống kê điền thiếu đều được tính từ train, không dùng validation/test.
    imputation_values = fit_imputation_values(
        frames["train"], config["numeric"], config["categorical"]
    )
    IMPUTATION_VALUES[source_name] = imputation_values
    PREPARED_FRAMES[source_name] = {
        split_name: apply_imputation(frame, imputation_values)
        for split_name, frame in frames.items()
    }

    required_columns = config["numeric"] + config["categorical"]
    for split_name, frame in PREPARED_FRAMES[source_name].items():
        assert frame[required_columns].isna().sum().sum() == 0, (
            f"{source_name}/{split_name}: vẫn còn giá trị thiếu trong trường dùng cho luật."
        )

print("Đã xử lý giá trị thiếu bằng thống kê từ tập huấn luyện.")

## Định nghĩa bộ luật và trọng số

Trọng số 3 thể hiện tín hiệu rất mạnh, trọng số 2 thể hiện tín hiệu mạnh và trọng số 1 là yếu tố bổ sung. Các mức của cùng một yếu tố loại trừ lẫn nhau để tránh cộng điểm hai lần.

In [ ]:
# Bảng này là tài liệu có cấu trúc của toàn bộ điều kiện được dùng khi chấm điểm.
RULES_TABLE = pd.DataFrame(
    [
        {"data_source": "student_dropout_and_success", "factor": "Tỷ lệ môn đạt học kỳ 2", "condition": "< 0.50", "points": 3},
        {"data_source": "student_dropout_and_success", "factor": "Tỷ lệ môn đạt học kỳ 2", "condition": "0.50 đến < 0.75", "points": 1},
        {"data_source": "student_dropout_and_success", "factor": "Điểm học kỳ 2", "condition": "< 10", "points": 2},
        {"data_source": "student_dropout_and_success", "factor": "Điểm học kỳ 2", "condition": "10 đến < 12", "points": 1},
        {"data_source": "student_dropout_and_success", "factor": "Tỷ lệ môn đạt học kỳ 1", "condition": "< 0.50", "points": 2},
        {"data_source": "student_dropout_and_success", "factor": "Tỷ lệ môn đạt học kỳ 1", "condition": "0.50 đến < 0.75", "points": 1},
        {"data_source": "student_dropout_and_success", "factor": "Học phí", "condition": "Chưa đóng đúng hạn", "points": 2},
        {"data_source": "student_dropout_and_success", "factor": "Công nợ", "condition": "Có nợ", "points": 1},
        {"data_source": "student_dropout_and_success", "factor": "Không tham gia đánh giá học kỳ 2", "condition": ">= 2 môn", "points": 1},
        {"data_source": "student_dropout", "factor": "GPA", "condition": "< 2.0", "points": 3},
        {"data_source": "student_dropout", "factor": "GPA", "condition": "2.0 đến < 2.5", "points": 1},
        {"data_source": "student_dropout", "factor": "Tỷ lệ chuyên cần", "condition": "< 75%", "points": 2},
        {"data_source": "student_dropout", "factor": "Tỷ lệ chuyên cần", "condition": "75% đến < 85%", "points": 1},
        {"data_source": "student_dropout", "factor": "Mức độ căng thẳng", "condition": ">= 7", "points": 2},
        {"data_source": "student_dropout", "factor": "Mức độ căng thẳng", "condition": "5 đến < 7", "points": 1},
        {"data_source": "student_dropout", "factor": "Thời gian tự học", "condition": "< 2 giờ/ngày", "points": 1},
        {"data_source": "student_dropout", "factor": "Nộp bài trễ", "condition": ">= 3 ngày", "points": 1},
        {"data_source": "student_dropout", "factor": "Truy cập Internet", "condition": "Không có", "points": 1},
        {"data_source": "student_dropout", "factor": "Công việc làm thêm", "condition": "Có", "points": 1},
    ]
)

print(RULES_TABLE.to_string(index=False))

## Kiểm tra căn cứ của luật trên tập huấn luyện

Mỗi điều kiện chính được đối chiếu với tỷ lệ bỏ học trong train set. `chenh_lech_phan_tram` dương cho biết nhóm thỏa điều kiện có tỷ lệ bỏ học cao hơn nhóm còn lại. Bảng này chỉ có vai trò giải thích căn cứ của luật, không sử dụng validation hoặc test.

In [ ]:
def add_academic_ratios(frame):
    """Tạo tỷ lệ môn đạt và xử lý trường hợp số môn đăng ký bằng 0."""

    result = frame.copy()
    for semester in ["1st", "2nd"]:
        enrolled = result[f"Curricular units {semester} sem (enrolled)"]
        approved = result[f"Curricular units {semester} sem (approved)"]
        # Không đăng ký môn được xem là tỷ lệ đạt bằng 0 trong bối cảnh cảnh báo bỏ học.
        result[f"approval_ratio_{semester}"] = (
            approved / enrolled.replace(0, np.nan)
        ).fillna(0.0)
    return result


source_1_train_audit = add_academic_ratios(
    PREPARED_FRAMES["student_dropout_and_success"]["train"]
)
source_2_train_audit = PREPARED_FRAMES["student_dropout"]["train"]

AUDIT_CONDITIONS = {
    "student_dropout_and_success": {
        "Tỷ lệ môn đạt học kỳ 2 dưới 50%": source_1_train_audit["approval_ratio_2nd"] < 0.50,
        "Điểm học kỳ 2 dưới 10": source_1_train_audit["Curricular units 2nd sem (grade)"] < 10,
        "Tỷ lệ môn đạt học kỳ 1 dưới 50%": source_1_train_audit["approval_ratio_1st"] < 0.50,
        "Học phí chưa đóng đúng hạn": source_1_train_audit["Tuition fees up to date"] == 0,
        "Có công nợ": source_1_train_audit["Debtor"] == 1,
        "Không đánh giá ít nhất 2 môn ở học kỳ 2": source_1_train_audit["Curricular units 2nd sem (without evaluations)"] >= 2,
    },
    "student_dropout": {
        "GPA dưới 2.0": source_2_train_audit["GPA"] < 2.0,
        "Chuyên cần dưới 75%": source_2_train_audit["Attendance_Rate"] < 75,
        "Căng thẳng từ 7 trở lên": source_2_train_audit["Stress_Index"] >= 7,
        "Tự học dưới 2 giờ mỗi ngày": source_2_train_audit["Study_Hours_per_Day"] < 2,
        "Nộp bài trễ từ 3 ngày": source_2_train_audit["Assignment_Delay_Days"] >= 3,
        "Không có Internet": source_2_train_audit["Internet_Access"].str.lower() == "no",
        "Có công việc làm thêm": source_2_train_audit["Part_Time_Job"].str.lower() == "yes",
    },
}

TRAIN_AUDIT_FRAMES = {
    "student_dropout_and_success": source_1_train_audit,
    "student_dropout": source_2_train_audit,
}

audit_rows = []
for source_name, conditions in AUDIT_CONDITIONS.items():
    train_frame = TRAIN_AUDIT_FRAMES[source_name]
    for condition_name, condition_mask in conditions.items():
        # So sánh tỷ lệ bỏ học giữa nhóm thỏa và không thỏa điều kiện.
        risk_rate = train_frame.loc[condition_mask, "Dropout"].mean()
        other_rate = train_frame.loc[~condition_mask, "Dropout"].mean()
        audit_rows.append(
            {
                "data_source": source_name,
                "condition": condition_name,
                "so_dong_thoa_dieu_kien": int(condition_mask.sum()),
                "ty_le_bo_hoc_nhom_rui_ro": round(risk_rate * 100, 2),
                "ty_le_bo_hoc_nhom_con_lai": round(other_rate * 100, 2),
                "chenh_lech_phan_tram": round((risk_rate - other_rate) * 100, 2),
            }
        )

train_rule_evidence = pd.DataFrame(audit_rows)
assert (train_rule_evidence["chenh_lech_phan_tram"] > 0).all(), (
    "Có điều kiện luật không làm tăng tỷ lệ bỏ học trên train set."
)
print(train_rule_evidence.to_string(index=False))

## Hàm chấm điểm cho Nguồn 1

Nguồn 1 tập trung vào kết quả học kỳ và tình trạng tài chính. Tổng điểm tối đa là 11.

In [ ]:
def calculate_source_1_risk(row):
    """Tính điểm rủi ro từ dữ liệu học vụ và tài chính."""

    score = 0
    factors = []
    recommendations = []

    # Tính tỷ lệ môn đạt; nếu không đăng ký môn thì tỷ lệ được xem là 0.
    enrolled_1 = row["Curricular units 1st sem (enrolled)"]
    enrolled_2 = row["Curricular units 2nd sem (enrolled)"]
    approval_ratio_1 = (
        row["Curricular units 1st sem (approved)"] / enrolled_1
        if enrolled_1 > 0 else 0.0
    )
    approval_ratio_2 = (
        row["Curricular units 2nd sem (approved)"] / enrolled_2
        if enrolled_2 > 0 else 0.0
    )

    # Tỷ lệ môn đạt học kỳ 2 là tín hiệu mạnh nhất của bộ luật.
    if approval_ratio_2 < 0.50:
        score += 3
        factors.append("Tỷ lệ môn đạt học kỳ 2 dưới 50%")
        recommendations.append("Tư vấn học vụ và lập kế hoạch học lại các môn chưa đạt")
    elif approval_ratio_2 < 0.75:
        score += 1
        factors.append("Tỷ lệ môn đạt học kỳ 2 dưới 75%")
        recommendations.append("Theo dõi tiến độ hoàn thành môn học trong học kỳ tiếp theo")

    # Điểm học kỳ 2 thấp phản ánh khó khăn học tập gần thời điểm đánh giá.
    grade_2 = row["Curricular units 2nd sem (grade)"]
    if grade_2 < 10:
        score += 2
        factors.append("Điểm trung bình học kỳ 2 dưới 10")
        recommendations.append("Bố trí hỗ trợ học tập cho các môn có kết quả thấp")
    elif grade_2 < 12:
        score += 1
        factors.append("Điểm trung bình học kỳ 2 dưới 12")
        recommendations.append("Theo dõi kết quả học tập trong học kỳ tiếp theo")

    # Học kỳ 1 cung cấp thêm bằng chứng về khó khăn kéo dài.
    if approval_ratio_1 < 0.50:
        score += 2
        factors.append("Tỷ lệ môn đạt học kỳ 1 dưới 50%")
        recommendations.append("Rà soát các môn nền tảng chưa đạt")
    elif approval_ratio_1 < 0.75:
        score += 1
        factors.append("Tỷ lệ môn đạt học kỳ 1 dưới 75%")
        recommendations.append("Theo dõi việc hoàn thành các môn còn thiếu")

    # Các trở ngại tài chính được chấm riêng vì cần hình thức can thiệp khác.
    if row["Tuition fees up to date"] == 0:
        score += 2
        factors.append("Học phí chưa được đóng đúng hạn")
        recommendations.append("Liên hệ tư vấn học phí và phương án hỗ trợ tài chính")
    if row["Debtor"] == 1:
        score += 1
        factors.append("Sinh viên đang có công nợ")
        recommendations.append("Rà soát tình trạng công nợ và kế hoạch thanh toán")

    # Việc không tham gia đánh giá có thể phản ánh dấu hiệu rời bỏ quá trình học.
    if row["Curricular units 2nd sem (without evaluations)"] >= 2:
        score += 1
        factors.append("Không tham gia đánh giá ít nhất 2 môn ở học kỳ 2")
        recommendations.append("Liên hệ xác minh nguyên nhân không tham gia đánh giá")

    if not factors:
        factors.append("Không phát hiện yếu tố rủi ro đáng kể")
        recommendations.append("Tiếp tục theo dõi định kỳ")

    return pd.Series(
        {
            "raw_score": int(score),
            "risk_factors": factors,
            "recommendations": recommendations,
        }
    )

## Hàm chấm điểm cho Nguồn 2

Nguồn 2 tập trung vào GPA, chuyên cần, căng thẳng và hành vi học tập. Tổng điểm tối đa là 11.

In [ ]:
def calculate_source_2_risk(row):
    """Tính điểm rủi ro từ hành vi và điều kiện học tập."""

    score = 0
    factors = []
    recommendations = []

    # GPA có trọng số cao nhất vì cho thấy chênh lệch rủi ro rõ trên train set.
    if row["GPA"] < 2.0:
        score += 3
        factors.append("GPA dưới 2.0")
        recommendations.append("Tư vấn học tập và lập kế hoạch cải thiện GPA")
    elif row["GPA"] < 2.5:
        score += 1
        factors.append("GPA từ 2.0 đến dưới 2.5")
        recommendations.append("Theo dõi kết quả học tập trong học kỳ tiếp theo")

    # Chuyên cần thấp là tín hiệu sinh viên giảm gắn kết với quá trình học.
    if row["Attendance_Rate"] < 75:
        score += 2
        factors.append("Tỷ lệ chuyên cần dưới 75%")
        recommendations.append("Liên hệ sinh viên và xác định nguyên nhân nghỉ học")
    elif row["Attendance_Rate"] < 85:
        score += 1
        factors.append("Tỷ lệ chuyên cần từ 75% đến dưới 85%")
        recommendations.append("Theo dõi chuyên cần và nhắc nhở đi học đầy đủ")

    # Căng thẳng cao cần được kết hợp với hỗ trợ tâm lý.
    if row["Stress_Index"] >= 7:
        score += 2
        factors.append("Mức độ căng thẳng cao")
        recommendations.append("Đề xuất tư vấn tâm lý và theo dõi sức khỏe tinh thần")
    elif row["Stress_Index"] >= 5:
        score += 1
        factors.append("Mức độ căng thẳng cần theo dõi")
        recommendations.append("Theo dõi mức độ căng thẳng định kỳ")

    # Các yếu tố hành vi bổ sung được cộng một điểm khi xuất hiện.
    if row["Study_Hours_per_Day"] < 2:
        score += 1
        factors.append("Thời gian tự học dưới 2 giờ mỗi ngày")
        recommendations.append("Hỗ trợ xây dựng thời gian biểu học tập")
    if row["Assignment_Delay_Days"] >= 3:
        score += 1
        factors.append("Nộp bài trễ từ 3 ngày trở lên")
        recommendations.append("Nhắc hạn bài và hỗ trợ kỹ năng quản lý thời gian")
    if str(row["Internet_Access"]).strip().lower() == "no":
        score += 1
        factors.append("Không có điều kiện truy cập Internet")
        recommendations.append("Hỗ trợ thiết bị hoặc địa điểm truy cập Internet")
    if str(row["Part_Time_Job"]).strip().lower() == "yes":
        score += 1
        factors.append("Công việc làm thêm có thể ảnh hưởng việc học")
        recommendations.append("Tư vấn cân bằng thời gian làm thêm và học tập")

    if not factors:
        factors.append("Không phát hiện yếu tố rủi ro đáng kể")
        recommendations.append("Tiếp tục theo dõi định kỳ")

    return pd.Series(
        {
            "raw_score": int(score),
            "risk_factors": factors,
            "recommendations": recommendations,
        }
    )

## Chấm điểm ba tập dữ liệu

Điểm thô được chuẩn hóa bằng công thức `raw_score / max_score`. Giá trị chuẩn hóa chỉ dùng để xếp hạng và so sánh; không được gọi là xác suất bỏ học.

In [ ]:
SCORING_FUNCTIONS = {
    "student_dropout_and_success": calculate_source_1_risk,
    "student_dropout": calculate_source_2_risk,
}


def score_frame(frame, scoring_function, max_score):
    """Áp dụng một bộ luật và chuẩn hóa điểm rủi ro về khoảng 0–1."""

    scored_columns = frame.apply(scoring_function, axis=1)
    scored = pd.concat([frame.reset_index(drop=True), scored_columns], axis=1)
    scored["prediction_score"] = scored["raw_score"] / max_score
    assert scored["raw_score"].between(0, max_score).all()
    assert scored["prediction_score"].between(0, 1).all()
    return scored


SCORED_FRAMES = {}
for source_name, frames in PREPARED_FRAMES.items():
    max_score = RULE_INPUTS[source_name]["max_score"]
    SCORED_FRAMES[source_name] = {
        split_name: score_frame(
            frame, SCORING_FUNCTIONS[source_name], max_score
        )
        for split_name, frame in frames.items()
    }
    print(f"Đã chấm điểm {source_name}.")

## Chọn ngưỡng cảnh báo trên tập xác thực

Notebook thử các ngưỡng điểm nguyên từ 1 đến điểm tối đa và chọn ngưỡng có F2-score cao nhất. Nếu F2 bằng nhau, Recall rồi Precision được ưu tiên. Tập kiểm thử không tham gia quá trình này.

In [ ]:
def select_alert_threshold(y_true, raw_score, max_score):
    """Chọn ngưỡng điểm cảnh báo bằng F2 trên tập xác thực."""

    rows = []
    for threshold in range(1, max_score + 1):
        prediction = (raw_score >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "precision": precision_score(
                    y_true, prediction, zero_division=0
                ),
                "recall": recall_score(y_true, prediction, zero_division=0),
                "f2": fbeta_score(
                    y_true, prediction, beta=2, zero_division=0
                ),
            }
        )

    threshold_table = pd.DataFrame(rows)
    selected = threshold_table.sort_values(
        ["f2", "recall", "precision", "threshold"],
        ascending=[False, False, False, False],
    ).iloc[0]
    return int(selected["threshold"]), threshold_table


def calculate_metrics(y_true, score, raw_score, threshold):
    """Tính metric phân loại và ma trận nhầm lẫn tại một ngưỡng điểm."""

    prediction = (raw_score >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        y_true, prediction, labels=[0, 1]
    ).ravel()
    return {
        "threshold_raw": int(threshold),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "f2": fbeta_score(y_true, prediction, beta=2, zero_division=0),
        "accuracy": accuracy_score(y_true, prediction),
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": average_precision_score(y_true, score),
        "tn": int(true_negative),
        "fp": int(false_positive),
        "fn": int(false_negative),
        "tp": int(true_positive),
    }


SELECTED_THRESHOLDS = {}
HIGH_THRESHOLDS = {}
THRESHOLD_TABLES = {}
validation_rows = []

for source_name, frames in SCORED_FRAMES.items():
    validation_frame = frames["validation"]
    max_score = RULE_INPUTS[source_name]["max_score"]
    threshold, threshold_table = select_alert_threshold(
        validation_frame["Dropout"], validation_frame["raw_score"], max_score
    )
    # Mức Cao bắt đầu từ 6 điểm hoặc cao hơn ngưỡng cảnh báo ít nhất 2 điểm.
    high_threshold = min(max_score, max(6, threshold + 2))
    SELECTED_THRESHOLDS[source_name] = threshold
    HIGH_THRESHOLDS[source_name] = high_threshold
    THRESHOLD_TABLES[source_name] = threshold_table

    metrics = calculate_metrics(
        validation_frame["Dropout"],
        validation_frame["prediction_score"],
        validation_frame["raw_score"],
        threshold,
    )
    metrics.update(
        {
            "data_source": source_name,
            "solution_type": "rule_based",
            "high_threshold_raw": high_threshold,
        }
    )
    validation_rows.append(metrics)

validation_metrics = pd.DataFrame(validation_rows)
print("Kết quả chọn ngưỡng trên tập xác thực:")
print(validation_metrics.round(4).to_string(index=False))

## Đánh giá cuối cùng trên tập kiểm thử

Ngưỡng đã chốt trên validation được áp dụng nguyên trạng cho test. `low` là dưới ngưỡng cảnh báo, `medium` là từ ngưỡng cảnh báo đến dưới ngưỡng cao, và `high` là từ ngưỡng cao trở lên.

In [ ]:
test_rows = []
prediction_outputs = {}
risk_level_rows = []

for source_name, frames in SCORED_FRAMES.items():
    test_frame = frames["test"].copy()
    threshold = SELECTED_THRESHOLDS[source_name]
    high_threshold = HIGH_THRESHOLDS[source_name]

    # Áp dụng ngưỡng validation mà không điều chỉnh theo kết quả test.
    test_frame["y_pred"] = (test_frame["raw_score"] >= threshold).astype(int)
    test_frame["risk_level"] = np.select(
        [
            test_frame["raw_score"] < threshold,
            test_frame["raw_score"] < high_threshold,
        ],
        ["low", "medium"],
        default="high",
    )

    metrics = calculate_metrics(
        test_frame["Dropout"],
        test_frame["prediction_score"],
        test_frame["raw_score"],
        threshold,
    )
    metrics.update(
        {
            "data_source": source_name,
            "solution_type": "rule_based",
            "high_threshold_raw": high_threshold,
        }
    )
    test_rows.append(metrics)

    # Danh sách nguyên nhân và khuyến nghị được mã hóa JSON để đọc lại an toàn từ CSV.
    prediction_outputs[source_name] = pd.DataFrame(
        {
            "row_id": test_frame["row_id"],
            "y_true": test_frame["Dropout"].astype(int),
            "prediction_score": test_frame["prediction_score"],
            "y_pred": test_frame["y_pred"],
            "raw_score": test_frame["raw_score"],
            "risk_level": test_frame["risk_level"],
            "risk_factors": test_frame["risk_factors"].apply(
                lambda value: json.dumps(value, ensure_ascii=False)
            ),
            "recommendations": test_frame["recommendations"].apply(
                lambda value: json.dumps(value, ensure_ascii=False)
            ),
            "alert_threshold_raw": threshold,
            "high_threshold_raw": high_threshold,
        }
    ).sort_values("row_id")

    # Thống kê tỷ lệ bỏ học theo mức rủi ro chỉ để báo cáo, không dùng để chỉnh luật.
    level_summary = (
        test_frame.groupby("risk_level", as_index=False)
        .agg(
            student_count=("row_id", "size"),
            dropout_count=("Dropout", "sum"),
            dropout_rate=("Dropout", "mean"),
        )
    )
    level_summary.insert(0, "data_source", source_name)
    risk_level_rows.append(level_summary)

test_metrics = pd.DataFrame(test_rows)
risk_level_summary = pd.concat(risk_level_rows, ignore_index=True)

print("Kết quả cuối cùng trên tập kiểm thử:")
print(test_metrics.round(4).to_string(index=False))
print("\nTỷ lệ bỏ học theo mức rủi ro:")
print(risk_level_summary.round(4).to_string(index=False))

## Lưu cấu hình, metric và file dự đoán

Cấu hình lưu ngưỡng, điểm tối đa và giá trị điền thiếu để quy trình có thể tái lập. Hai file dự đoán có cùng bốn cột lõi với kết quả Machine Learning và bổ sung phần giải thích quyết định.

In [ ]:
# Lưu tài liệu luật và bằng chứng được tính từ train set.
RULES_TABLE.to_csv(
    RULE_DIR / "rule_definitions.csv", index=False, encoding="utf-8-sig"
)
train_rule_evidence.to_csv(
    RULE_DIR / "train_rule_evidence.csv", index=False, encoding="utf-8-sig"
)
risk_level_summary.to_csv(
    RULE_DIR / "test_risk_level_summary.csv", index=False, encoding="utf-8-sig"
)

# Lưu bảng metric validation và test để dùng trong phần so sánh giải pháp.
validation_metrics.to_csv(
    PROJECT_ROOT / "outputs" / "rule_based_validation_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
test_metrics.to_csv(
    PROJECT_ROOT / "outputs" / "rule_based_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

for source_name, prediction_frame in prediction_outputs.items():
    prediction_frame.to_csv(
        PREDICTION_DIR / f"{source_name}_rule_based.csv",
        index=False,
        encoding="utf-8-sig",
    )

# Chuyển toàn bộ giá trị cấu hình về kiểu Python chuẩn trước khi ghi JSON.
rule_config = {}
for source_name in SOURCE_FRAMES:
    rule_config[source_name] = {
        "max_score": int(RULE_INPUTS[source_name]["max_score"]),
        "alert_threshold_raw": int(SELECTED_THRESHOLDS[source_name]),
        "high_threshold_raw": int(HIGH_THRESHOLDS[source_name]),
        "score_normalization": "raw_score / max_score",
        "prediction_score_is_probability": False,
        "imputation_values_from_train": {
            key: value.item() if hasattr(value, "item") else value
            for key, value in IMPUTATION_VALUES[source_name].items()
        },
    }

with (RULE_DIR / "rule_based_config.json").open("w", encoding="utf-8") as file:
    json.dump(rule_config, file, ensure_ascii=False, indent=2)

print("Đã lưu cấu hình và kết quả Rule-based Scoring vào outputs.")

## Kiểm tra đầu ra và tính công bằng khi so sánh

Phần cuối kiểm tra cấu trúc file, miền điểm và bảo đảm Rule-based có đúng cùng `row_id` với Machine Learning trên mỗi nguồn.

In [ ]:
required_columns = {
    "row_id", "y_true", "prediction_score", "y_pred",
    "risk_level", "risk_factors", "recommendations",
}

for source_name, prediction_frame in prediction_outputs.items():
    # Kiểm tra file Rule-based có đầy đủ trường và không có row_id trùng.
    assert required_columns.issubset(prediction_frame.columns)
    assert prediction_frame["row_id"].is_unique
    assert prediction_frame["prediction_score"].between(0, 1).all()
    assert set(prediction_frame["y_pred"].unique()).issubset({0, 1})
    assert set(prediction_frame["risk_level"].unique()).issubset(
        {"low", "medium", "high"}
    )

    # Đối chiếu trực tiếp với file ML để bảo đảm so sánh trên cùng test set.
    ml_path = PREDICTION_DIR / f"{source_name}_ml.csv"
    if not ml_path.exists():
        raise FileNotFoundError(
            f"Thiếu {ml_path.name}; hãy chạy notebook Machine Learning trước."
        )
    ml_prediction = pd.read_csv(ml_path)
    assert set(prediction_frame["row_id"]) == set(ml_prediction["row_id"]), (
        f"{source_name}: Rule-based và ML không dùng cùng row_id kiểm thử."
    )
    assert dict(zip(prediction_frame["row_id"], prediction_frame["y_true"])) == dict(
        zip(ml_prediction["row_id"], ml_prediction["y_true"])
    ), f"{source_name}: y_true của Rule-based và ML không khớp."

print("Kiểm tra hoàn tất: Rule-based và ML sử dụng cùng test set cho cả hai nguồn.")

## Kết luận

Hai nguồn đã có bộ luật riêng, ngưỡng cảnh báo được chọn trên validation và kết quả cuối cùng được tạo trên đúng test set của Machine Learning. Bước tiếp theo là tổng hợp metric định lượng và tiêu chí định tính để so sánh hai loại giải pháp trên từng nguồn.